# Day 91: One command that runs the whole check

**Goal:** until now my checks lived in notebooks. A **script** lets me, or anyone, run the whole check with one command.

**The picture:** a notebook is a kitchen where I cook and taste as I go. A script is the **recipe card that cooks the whole meal by itself** when I press one button.

## 1. Five ideas
1. **A script is a `.py` file** that runs top to bottom with `python filename.py` in the terminal.
2. **Put the work in `main()` and end the file with:**
   ```python
   if __name__ == "__main__":
       main()
   ```
   Python sets a label `__name__` automatically. When I run a file directly, the label is `"__main__"`. When another file *imports* it, the label is the file's own name, so nothing runs. Like a toolbox in a garage: when a neighbour borrows a hammer, the whole garage does not start working.
3. **The import problem disappears.** Python looks in the **script's own folder first**. `run_checks.py` sits in `csa_soil_project/`, next to `agronomic_functions.py` and `db_config.py`, so no `sys.path` lines are needed.
4. **A script knows where it lives:** `Path(__file__).parent` is the folder of the script. Saving to `"Reports/..."` depends on where I stand in the terminal. `HERE / "Reports" / ...` always lands in the right place. It is like writing my home address on a parcel instead of "the shop near me".
5. **An exit code tells other programs whether the run worked.** `raise SystemExit(1)` ends the script with code **1** (failed). Success is code **0**.

## 2. Exercise (made-up data, run in the terminal)
`hello_report.py`:
```python
import pandas as pd

def main():
    toy = pd.DataFrame({"a": [1, None, 3]})
    print("rows:", len(toy))
    print("missing a:", int(toy["a"].isna().sum()))

if __name__ == "__main__":
    main()
```
`borrow.py`:
```python
from hello_report import main
```

| Step | Prediction | What happened |
|---|---|---|
| `python hello_report.py` | prints `rows: 3` and `missing a: 1` | as predicted |
| `python borrow.py` | prints nothing (imported, not the main file) | as predicted |
| `borrow.py` with `main()` added at the bottom | prints both lines (I call the function myself) | as predicted |

**Check question: why `if __name__ == "__main__":`?** It is a guard. When I run the file directly it executes, and when another file only imports it, it stays quiet.

## 3. The project script
Reference copy of `csa_soil_project/run_checks.py` (**your file is the source of truth**). It:
1. builds the flat table and the flags
2. merges `dataset_id`, with a **row-count self-check** (merges are where rows get lost or doubled)
3. prints the cleaning report
4. runs `dataset_alarm` on `orgc`, `ph`, `clay` and `sand`
5. prints the number of hard and soft flags
6. saves the report to `Reports/run_checks_report.csv`
7. stops with `CHECK FAILED: ...` and exit code 1 if a `ValueError` happens

```python
"""Run all quality checks on the Ghana WoSIS soil table and save the report.

Usage (from the csa_soil_project folder):
    python run_checks.py
Exit code 0 means success, 1 means a check failed.
"""
from pathlib import Path
import pandas as pd
from agronomic_functions import build_flat, quality_flags, cleaning_report, dataset_alarm
from db_config import run_query


def main():
    """Build the table, run every check, print the results and save the report."""
    print("=== STARTING SOIL CHECKS ===")

    flat = build_flat("orgc", ["ph", "clay", "sand"])
    print(f"flat table shape: {flat.shape}")

    checked = quality_flags(flat)

    meta = run_query("SELECT dataset_id, layer_id FROM wosis_orgc WHERE country_name = 'Ghana'")
    traced = pd.merge(checked, meta, on="layer_id", how="left")

    if len(traced) != len(checked):
        raise ValueError(
            f"Merge changed row count! Before: {len(checked)}, After: {len(traced)}")
    print(f"Merged rows: {len(traced)}")

    report = cleaning_report(traced, ["ph", "clay", "sand"], key="layer_id")
    print("\n--- CLEANING REPORT ---")
    print(report.to_string(index=False))

    print("\n--- DATASET ALARMS ---")
    for col in ["orgc", "ph", "clay", "sand"]:
        alarm = dataset_alarm(traced, col)
        if len(alarm) == 0:
            print(f"{col}: OK (no flags)")
        else:
            print(f"{col}: FLAGGED")
            for name, value in alarm.items():
                print(f"  {name} = {value:.1f}")

    hard_cols = ["flag_ph", "flag_clay_sand", "flag_clay", "flag_sand", "flag_oc_neg"]
    print(f"\nHard flags: {traced[hard_cols].sum().sum()}")
    print(f"Soft flags: {traced['flag_oc_high'].sum()}")

    HERE = Path(__file__).parent
    out_path = HERE / "Reports" / "run_checks_report.csv"
    report.to_csv(out_path, index=False)
    print(f"\nReport saved to: {out_path}")


if __name__ == "__main__":
    try:
        main()
    except ValueError as e:
        print("CHECK FAILED:", e)
        raise SystemExit(1)
```

## 4. Run it from this notebook and read the exit code
Expected: the report numbers from Days 89 and 90, `Hard flags: 0`, `Soft flags: 144`, and **exit code 0**. (In the terminal, `echo $LASTEXITCODE` gives the same number.)

In [ ]:
import subprocess, sys

result = subprocess.run([sys.executable, "../../run_checks.py"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)
print("exit code:", result.returncode)

## 5. Proof the safety net works
I temporarily asked `cleaning_report` for a column that does not exist (`"ph2"`). The script printed `CHECK FAILED: Column ph2 not found in table.` and the exit code was **1**. Then I changed it back, and the exit code was **0**.

**Check question: why build the save path from `Path(__file__).parent`?**
It guarantees the report always lands in the folder where the script lives, no matter where I run the script from.

## 6. What I keep from today
- `if __name__ == "__main__":` runs the work only when the file is run directly.
- A script finds its own folder with `Path(__file__).parent`.
- Exit code 0 means success, 1 means failure. `SystemExit(1)` sets it.
- Python looks in the script's own folder first, so the import problem goes away.
- File names must match exactly (`run_checks.py`, not `run_check.py`).
- Docstrings: a module docstring, and one line under `main()`.

**Next:** Day 92, the first chart.